# Phase 3: Evaluate on Standard LIBERO + LIBERO-PRO

Evaluate the trained SmolVLA model on:
1. **Standard LIBERO**: 10 tasks × 50 episodes = 500 trials (task success)
2. **LIBERO-PRO**: Same tasks with perturbations (robustness/generalization)

**The key insight:** Models scoring >90% on standard LIBERO collapse to 0% on LIBERO-PRO.
Any non-zero LIBERO-PRO score is meaningful progress.

**Runtime:** GPU (T4 sufficient for inference)

## 1. Setup

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev ffmpeg > /dev/null 2>&1
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

pip install -q robosuite==1.4.1 robomimic==0.2.0 bddl==1.0.1
pip install -q hydra-core easydict einops cloudpickle "gym==0.25.2"
pip install -q imageio[ffmpeg] matplotlib seaborn pandas tqdm rich

if [ ! -d "LIBERO" ]; then git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git; fi
pip install -q -e LIBERO/

if [ ! -d "lerobot" ]; then git clone https://github.com/huggingface/lerobot.git; fi
cd lerobot && pip install -q -e ".[smolvla]"

pip install -q "numpy>=2.0,<2.1"
echo "Done"

In [ ]:
import os
os.kill(os.getpid(), 9)  # Restart runtime

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated\n")

import numpy as np
import torch
import json
import csv
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import base64

# Monkey-patch torch.load for LIBERO compatibility — only once
# (PyTorch 2.6+ defaults to weights_only=True but LIBERO init states use numpy)
if not hasattr(torch, "_original_load"):
    torch._original_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return torch._original_load(*args, **kwargs)
    torch.load = _patched_torch_load

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

print(f"robosuite: {robosuite.__version__}, numpy: {np.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print("torch.load patched for LIBERO compat")
print("Ready!")

## 2. Load Model

In [ ]:
# Set checkpoint path — either local or HuggingFace repo ID
# CHECKPOINT = "outputs/smolvla_libero_spatial/checkpoints/100000/pretrained_model"  # Local path
CHECKPOINT = "HuggingFaceVLA/smolvla_libero"  # HF's pre-trained checkpoint
SUITE = "libero_spatial"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_EPISODES = 50  # Per task (standard: 50)
MAX_STEPS = 600

# Load policy + preprocessor/postprocessor
policy = None
preprocess = None
postprocess = None
try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from lerobot.policies.factory import make_pre_post_processors

    policy = SmolVLAPolicy.from_pretrained(CHECKPOINT)
    policy.to(DEVICE)
    policy.eval()

    # Load preprocessor/postprocessor from saved model configs.
    # Handles: language tokenization, state normalization, action unnormalization.
    preprocess, postprocess = make_pre_post_processors(
        policy.config,
        CHECKPOINT,
        preprocessor_overrides={"device_processor": {"device": str(DEVICE)}},
    )

    print(f"Loaded SmolVLA from {CHECKPOINT}")
    print(f"  Preprocessor steps: {[type(s).__name__ for s in preprocess.steps]}")
    print(f"  Postprocessor steps: {[type(s).__name__ for s in postprocess.steps]}")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"Could not load model: {e}")
    print("Using RANDOM POLICY for testing the evaluation pipeline")

## 3. Evaluation Helpers

In [ ]:
from PIL import Image as PILImage


def get_action(policy, preprocess, postprocess, obs, task_language, device="cuda"):
    """Get action from SmolVLA using the proper LeRobot preprocessor/postprocessor pipeline.

    Pipeline:
      1. Build raw observation dict with LeRobot-compatible keys
      2. Convert to tensors, normalize images to [0,1], add batch dim
      3. Preprocess: tokenize language, normalize state (MEAN_STD)
      4. Model inference via select_action
      5. Postprocess: unnormalize action (MEAN_STD)
    """
    if policy is None:
        return np.random.uniform(-0.3, 0.3, size=7)

    from lerobot.policies.utils import prepare_observation_for_inference

    # Camera 1: agentview (main workspace camera)
    agentview = np.flip(
        obs.get("agentview_image", np.zeros((128, 128, 3), dtype=np.uint8)), axis=0
    ).copy()
    agentview = np.array(PILImage.fromarray(agentview).resize((256, 256)))

    # Camera 2: wrist / eye-in-hand camera
    wrist = obs.get("robot0_eye_in_hand_image", None)
    if wrist is not None:
        wrist = np.flip(wrist, axis=0).copy()
        wrist = np.array(PILImage.fromarray(wrist).resize((256, 256)))
    else:
        wrist = np.zeros((256, 256, 3), dtype=np.uint8)

    # Build state from actual robot proprioception
    eef_pos = obs.get("robot0_eef_pos", np.zeros(3))
    eef_quat = obs.get("robot0_eef_quat", np.zeros(4))
    state = np.concatenate([eef_pos, eef_quat]).astype(np.float32)

    # Raw observation dict with LeRobot keys (numpy arrays)
    raw_obs = {
        "observation.images.image": agentview,
        "observation.images.image2": wrist,
        "observation.state": state,
    }

    # Convert to tensors, normalize images to [0,1], add batch dim, move to device
    obs_frame = prepare_observation_for_inference(raw_obs, device, task=task_language)

    # Preprocess: tokenize language, normalize state (MEAN_STD)
    obs_preprocessed = preprocess(obs_frame)

    # Model inference
    with torch.no_grad():
        action = policy.select_action(obs_preprocessed)

    # Postprocess: unnormalize action (MEAN_STD)
    action = postprocess(action)

    if isinstance(action, torch.Tensor):
        action = action.cpu().numpy().flatten()

    return np.clip(action[:7], -1, 1)


def create_env(suite_name, task_id):
    """Create LIBERO environment."""
    benchmark_dict = benchmark.get_benchmark_dict()
    task_suite = benchmark_dict[suite_name]()
    task = task_suite.get_task(task_id)
    bddl_file = os.path.join(
        get_libero_path("bddl_files"), task.problem_folder, task.bddl_file,
    )
    env = OffScreenRenderEnv(bddl_file_name=bddl_file, camera_heights=128, camera_widths=128)
    return env, task, task_suite


def show_gif(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{data}" width="256">'))


print("Helpers loaded (with preprocessor/postprocessor pipeline).")

## 4. Standard LIBERO Evaluation

10 tasks × 50 episodes = 500 trials. Reports per-task and overall success rate.

In [ ]:
benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[SUITE]()
n_tasks = task_suite.n_tasks

print(f"Evaluating: {SUITE} ({n_tasks} tasks x {N_EPISODES} episodes)")
print("=" * 60)

standard_results = []
total_successes = 0

for task_id in range(n_tasks):
    env, task, ts = create_env(SUITE, task_id)
    successes = 0

    for ep in tqdm(range(N_EPISODES), desc=f"Task {task_id}", leave=False):
        env.seed(ep)
        obs = env.reset()

        init_states = ts.get_task_init_states(task_id)
        if ep < len(init_states):
            env.set_init_state(init_states[ep])
            obs = env.reset()

        # Reset policy internal state at start of each episode
        if policy is not None:
            policy.reset()

        for step in range(MAX_STEPS):
            action = get_action(policy, preprocess, postprocess,
                                obs, task.language, device=DEVICE)
            obs, reward, done, info = env.step(action)
            if done:
                if reward > 0:
                    successes += 1
                break

    sr = successes / N_EPISODES
    standard_results.append({"task_id": task_id, "language": task.language,
                            "successes": successes, "success_rate": sr})
    total_successes += successes
    print(f"  Task {task_id}: {sr:.1%} ({successes}/{N_EPISODES}) -- {task.language}")
    env.close()

overall_sr = total_successes / (n_tasks * N_EPISODES)
print(f"\nOverall: {overall_sr:.1%} ({total_successes}/{n_tasks * N_EPISODES})")

## 5. Plot Standard Results

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

task_ids = [r["task_id"] for r in standard_results]
srs = [r["success_rate"] for r in standard_results]
colors = ["#2ecc71" if s > 0.7 else "#f39c12" if s > 0.3 else "#e74c3c" for s in srs]

bars = ax.bar(task_ids, srs, color=colors, edgecolor="white")
ax.axhline(y=overall_sr, color="navy", linestyle="--", alpha=0.7, label=f"Mean: {overall_sr:.1%}")
ax.set_xlabel("Task ID")
ax.set_ylabel("Success Rate")
ax.set_title(f"Standard LIBERO — {SUITE}\nOverall: {overall_sr:.1%}")
ax.set_ylim(0, 1.05)
ax.set_xticks(task_ids)
ax.legend()

for bar, r in zip(bars, standard_results):
    lang = r["language"][:25]
    ax.text(bar.get_x() + bar.get_width()/2, -0.06, lang,
           ha="center", va="top", fontsize=6, rotation=45)

plt.tight_layout()
plt.show()

## 6. LIBERO-PRO Evaluation

**The real test.** Same tasks but with perturbations applied:
- **Position**: Objects moved to different valid positions
- **Instruction**: Paraphrased task descriptions
- **Object**: Changed appearance/color/scale
- **Environment**: Different backgrounds/lighting

Current SOTA (OpenVLA, pi0) scores **0%** here. Any non-zero score is progress.

In [ ]:
# LIBERO-PRO: Position perturbation test
# We perturb by using different random seeds for initial state

N_PRO_EPISODES = 10  # Fewer episodes per perturbation variant
N_VARIANTS = 5       # Number of perturbation variants

pro_results = {}

for ptype in ["position", "instruction"]:
    print(f"\n{'='*60}")
    print(f"  LIBERO-PRO: {ptype} perturbation")
    print(f"{'='*60}")

    ptype_successes = 0
    ptype_trials = 0
    task_results = []

    for task_id in range(n_tasks):
        env, task, ts = create_env(SUITE, task_id)
        task_successes = 0
        task_trials = 0

        for variant in range(N_VARIANTS):
            for ep in range(N_PRO_EPISODES):
                # Perturbation: use offset seeds for different initial positions
                perturbed_seed = 10000 + variant * 1000 + ep
                env.seed(perturbed_seed)
                obs = env.reset()

                # Reset policy internal state at start of each episode
                if policy is not None:
                    policy.reset()

                # For instruction perturbation, modify the language
                if ptype == "instruction":
                    lang = task.language.replace("pick up", "grasp").replace("put", "place")
                else:
                    lang = task.language

                for step in range(MAX_STEPS):
                    action = get_action(policy, preprocess, postprocess,
                                        obs, lang, device=DEVICE)
                    obs, reward, done, info = env.step(action)
                    if done:
                        if reward > 0:
                            task_successes += 1
                        break
                task_trials += 1

        sr = task_successes / max(task_trials, 1)
        task_results.append({"task_id": task_id, "language": task.language,
                           "successes": task_successes, "trials": task_trials, "success_rate": sr})
        ptype_successes += task_successes
        ptype_trials += task_trials
        print(f"  Task {task_id}: {sr:.1%} ({task_successes}/{task_trials}) -- {task.language}")
        env.close()

    pro_sr = ptype_successes / max(ptype_trials, 1)
    pro_results[ptype] = {
        "success_rate": pro_sr,
        "successes": ptype_successes,
        "trials": ptype_trials,
        "tasks": task_results,
    }
    print(f"\n  {ptype} overall: {pro_sr:.1%}")

## 7. Results Summary + Comparison

In [ ]:
print("\n" + "=" * 60)
print(f"  EVALUATION REPORT — {SUITE}")
print("=" * 60)
print(f"  Model: {'SmolVLA (trained)' if policy else 'Random Policy (baseline)'}")
print(f"  Checkpoint: {CHECKPOINT}")
print()
print(f"  Standard LIBERO: {overall_sr:.1%}")
for ptype, res in pro_results.items():
    print(f"  LIBERO-PRO ({ptype}): {res['success_rate']:.1%}")

# Comparison with published results
print("\n  --- Published Reference ---")
print("  OpenVLA (7B):  Standard ~85%  |  PRO-position ~0%  |  PRO-combined 0%")
print("  pi0:           Standard >90%  |  PRO-position ~0%  |  PRO-combined 0%")
print(f"  Ours (SmolVLA): Standard {overall_sr:.1%}", end="")
for ptype, res in pro_results.items():
    print(f"  |  PRO-{ptype} {res['success_rate']:.1%}", end="")
print()

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))

labels = ["Standard"] + [f"PRO-{p}" for p in pro_results.keys()]
our_rates = [overall_sr] + [res["success_rate"] for res in pro_results.values()]

colors = ["#3498db"] + ["#e74c3c"] * len(pro_results)
bars = ax.bar(labels, our_rates, color=colors, edgecolor="white")

for bar, rate in zip(bars, our_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
           f"{rate:.1%}", ha="center", fontweight="bold")

ax.set_ylabel("Success Rate")
ax.set_title(f"SmolVLA — {SUITE}: Standard vs LIBERO-PRO")
ax.set_ylim(0, max(max(our_rates) * 1.3, 0.1))

plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
results = {
    "suite": SUITE,
    "checkpoint": CHECKPOINT,
    "model": "SmolVLA" if policy else "Random",
    "standard": {
        "overall_success_rate": overall_sr,
        "tasks": standard_results,
    },
    "libero_pro": {k: v for k, v in pro_results.items()},
}

os.makedirs("results", exist_ok=True)
with open(f"results/{SUITE}_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to results/{SUITE}_evaluation.json")

# CSV summary
with open(f"results/{SUITE}_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["eval_type", "success_rate", "trials"])
    writer.writerow(["standard", f"{overall_sr:.4f}", n_tasks * N_EPISODES])
    for ptype, res in pro_results.items():
        writer.writerow([f"pro_{ptype}", f"{res['success_rate']:.4f}", res["trials"]])

print(f"CSV saved to results/{SUITE}_summary.csv")

## Phase 3 Complete!

**Key metrics to track:**
- Standard LIBERO success rate (target: >80% for SmolVLA)
- LIBERO-PRO success rate (target: >0% — any non-zero is novel)
- Per-perturbation breakdown

**Next steps:**
- Try data augmentation during training to improve PRO scores
- Train on all LIBERO suites (not just Spatial)
- Compare against other models (if compute allows)